### Bài thực hành số 2. Sử dụng thư viện ML: Phân tích và Dự đoán Dữ liệu NYC Taxi
Bài thực hành này hướng dẫn từng bước quy trình Học máy (Machine Learning) trên PySpark qua bài toán thực tế với dữ liệu NYC Taxi Trips. Quá trình bao gồm 3 nhóm bài toán:
1. Gom cụm (KMeans): Phân nhóm các chuyến đi theo khoảng cách và giá cước.
2. Phân loại (Logistic Regression & Decision Tree): Dự đoán chuyến đi có nhận tiền thưởng cao (Tip cao) hay không.
3. Pipeline & Model Persistence: Tự động hóa quy trình sản xuất và lưu trữ mô hình.

In [13]:
from pyspark.sql import SparkSession

# Khởi tạo SparkSession tối ưu hóa bộ nhớ cho Colab
spark = SparkSession.builder \
    .appName("PySpark_ML_NYC_Taxi_Real") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

1 - Tải & Nạp dữ liệu Parquet đã làm sạch ở bài thực hành trước

In [14]:
path = "./nyc_data/yellow_tripdata_2023-01-02-parquet"
df_raw = spark.read.parquet(path)

print(f"Tổng số dòng ban đầu: {df_raw.count():,}")
print("\nCấu trúc Schema của tập dữ liệu:")
df_raw.printSchema()

print("\n5 dòng dữ liệu mẫu:")
df_raw.select("tpep_pickup_datetime", "passenger_count", "trip_distance", "payment_type", "fare_amount", "tip_amount").show(5)

Tổng số dòng ban đầu: 5,651,462

Cấu trúc Schema của tập dữ liệu:
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- speed_mph: 

2 - Khám phá & Làm sạch dữ liệu (EDA & Data Cleaning)

In [ ]:
from pyspark.sql.functions import col, hour, count, avg, round, when

# 1. Kiểm tra dữ liệu nhiễu/ngoại lệ trong dữ liệu
# Trong thực tế: Chuyến đi có khoảng cách <= 0, cước phí <= 0, hoặc giá trị NULL là dữ liệu rác.
df_clean = df_raw.filter(
    (col("trip_distance") > 0) & 
    (col("trip_distance") < 100) &          # Loại bỏ các chuyến đi ảo > 100 miles
    (col("fare_amount") > 0) & 
    (col("fare_amount") < 500) &            # Loại bỏ cước phí bất thường > 500$
    (col("passenger_count").isNotNull()) & 
    (col("passenger_count") > 0)
)

# 2. Tạo cột mới 'pickup_hour' trích xuất giờ từ 'tpep_pickup_datetime'
df_clean = df_clean.withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))

# 3. Lưu ý thực tế: Dữ liệu taxi NYC chỉ ghi nhận tiền Tip chính xác đối với hình thức thanh toán thẻ Thẻ Tín Dụng (payment_type = 1).
# Với thanh toán Tiền Mặt (payment_type = 2), lái xe tự nhận cash và hệ thống mặc định ghi tip = $0.
df_credit = df_clean.filter(col("payment_type") == 1)

print(f"Số lượng bản ghi sau khi làm sạch: {df_credit.count():,}")

# Thống kê giá cước và tiền tip trung bình theo khung giờ trong ngày
df_credit.groupBy("pickup_hour").agg(
    count("*").alias("so_chuyen"),
    round(avg("fare_amount"), 2).alias("cuoc_tb"),
    round(avg("tip_amount"), 2).alias("tip_tb")
).orderBy("pickup_hour").show(10)

Bài tập 1: Thống kê xem có bao nhiêu chuyến đi trong tập dữ liệu df_clean có hình thức thanh toán bằng Tiền mặt (payment_type == 2).
Tìm khoảng cách chuyến đi lớn nhất (max) và nhỏ nhất (min) trong tập df_clean.

3 - Tiền xử lý & Chọn thuộc tính (Feature Engineering & Selection)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1. Tạo nhãn nhị phân 'high_tip': 1 nếu tiền Tip > 20% tiền cước (fare_amount), ngược lại 0
df_labeled = df_credit.withColumn(
    "high_tip",
    when((col("tip_amount") / col("fare_amount")) > 0.20, 1).otherwise(0)
)

# 2. Lựa chọn các thuộc tính dự báo (Features)
feature_cols = ["trip_distance", "fare_amount", "passenger_count", "pickup_hour", "PULocationID", "DOLocationID"]

# 3. Gom nhóm các đặc trưng vào định dạng Vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
df_assembled = assembler.transform(df_labeled)

# 4. Chuẩn hóa dữ liệu (Feature Scaling) bằng StandardScaler
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_assembled)
df_prepared = scaler_model.transform(df_assembled)

df_prepared.select("raw_features", "features", "high_tip").show(3, truncate=False)

Bài tập 2: Hãy tính tỷ lệ phần trăm các chuyến đi đạt nhãn high_tip = 1 so với high_tip = 0 trong tập dữ liệu để kiểm tra xem tập dữ liệu có bị mất cân bằng nhãn (imbalanced) hay không.
Thêm cột RatecodeID vào danh sách feature_cols và xử lý các giá trị Null của cột này (nếu có) bằng phương thức .na.fill(1).

4 - Phân cụm các chuyến taxi bằng KMeans (Unsupervised Learning)

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Ý nghĩa thực tiễn: Gom nhóm các loại hình chuyến đi (Chuyến đi ngắn nội thành, Chuyến đi đường dài/sân bay, v.v.)
assembler_kmeans = VectorAssembler(
    inputCols=["trip_distance", "fare_amount"], 
    outputCol="kmeans_features"
)
df_kmeans_input = assembler_kmeans.transform(df_credit)

# Khởi tạo mô hình KMeans phân làm k = 3 cụm
kmeans = KMeans(featuresCol="kmeans_features", predictionCol="cluster_id", k=3, seed=42)
kmeans_model = kmeans.fit(df_kmeans_input)

# Dự đoán cụm cho dữ liệu
df_clustered = kmeans_model.transform(df_kmeans_input)

# In tọa độ tâm cụm (Khoảng cách TB, Cước phí TB)
print("--- TÂM CÁC CỤM CHUYẾN ĐI (trip_distance, fare_amount) ---")
for i, center in enumerate(kmeans_model.clusterCenters()):
    print(f"Cụm {i}: Khoảng cách trung bình = {center[0]:.2f} miles | Cước trung bình = ${center[1]:.2f}")

# Thống kê số lượng chuyến đi ở mỗi cụm
df_clustered.groupBy("cluster_id").count().show()

Bài tập 3: Sử dụng ClusteringEvaluator để tính toán chỉ số Silhouette Score (độ phân tách giữa các cụm) của mô hình vừa huấn luyện.
Thử nghiệm huấn luyện lại KMeans với k=4 cụm và đưa ra nhận xét về ý nghĩa thực tế của cụm mới xuất hiện.

5 - Phân loại bằng Logistic Regression (Supervised Learning)

In [ ]:
from pyspark.ml.classification import LogisticRegression

# Chia tập dữ liệu thành Train (80%) và Test (20%)
train_data, test_data = df_prepared.randomSplit([0.8, 0.2], seed=42)
train_data.cache() # Tối ưu tốc độ huấn luyện trong Spark

# Khởi tạo và huấn luyện mô hình Logistic Regression dự đoán chuyến đi có 'high_tip' hay không
lr = LogisticRegression(featuresCol="features", labelCol="high_tip", maxIter=15)
lr_model = lr.fit(train_data)

# Dự đoán trên tập Test
predictions_lr = lr_model.transform(test_data)
predictions_lr.select("features", "high_tip", "prediction", "probability").show(5)

Bài tập 4: Trích xuất mảng hệ số lr_model.coefficients và biến hệ số chặn lr_model.intercept.
Ghép tương ứng các hệ số đó với danh sách cột feature_cols để xác định yếu tố nào làm tăng khả năng nhận tiền Tip cao nhất.

6 - Phân loại bằng Decision Tree (Supervised Learning)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

# Khởi tạo mô hình Cây quyết định (Decision Tree)
dt = DecisionTreeClassifier(
    featuresCol="features", 
    labelCol="high_tip", 
    maxDepth=5, 
    seed=42
)

# Huấn luyện mô hình
dt_model = dt.fit(train_data)

# Dự đoán trên tập Test
predictions_dt = dt_model.transform(test_data)
predictions_dt.select("high_tip", "prediction", "probability").show(5)

Bài tập 5: In cấu trúc logic cây phân nhánh bằng câu lệnh print(dt_model.toDebugString).
Thay đổi độ sâu của cây maxDepth = 10 và tiến hành huấn luyện lại.

7 - Đánh giá & So sánh mô hình (Model Evaluation)

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Evaluator cho chỉ số AUC (Area Under ROC)
evaluator_auc = BinaryClassificationEvaluator(labelCol="high_tip", metricName="areaUnderROC")

# Evaluator cho chỉ số Accuracy, Precision, Recall
evaluator_acc = MulticlassClassificationEvaluator(labelCol="high_tip", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="high_tip", metricName="precisionByLabel")

# Đánh giá mô hình Logistic Regression
auc_lr = evaluator_auc.evaluate(predictions_lr)
acc_lr = evaluator_acc.evaluate(predictions_lr)

# Đánh giá mô hình Decision Tree
auc_dt = evaluator_auc.evaluate(predictions_dt)
acc_dt = evaluator_acc.evaluate(predictions_dt)

print("============ KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ============")
print(f"Logistic Regression -> AUC-ROC: {auc_lr:.4f} | Accuracy: {acc_lr:.4f}")
print(f"Decision Tree       -> AUC-ROC: {auc_dt:.4f} | Accuracy: {acc_dt:.4f}")

Bài tập 6: Sử dụng PySpark SQL để lập Confusion Matrix (Ma trận nhầm lẫn) giữa high_tip và prediction cho kết quả dự đoán của mô hình Decision Tree.
Dựa trên chỉ số AUC-ROC, cho biết mô hình nào hoạt động tốt hơn trên tập dữ liệu NYC Taxi này.

8 - Xây dựng quy trình tự động với PySpark Pipeline

In [ ]:
from pyspark.ml import Pipeline

# 1. Định nghĩa các bước (Stages) xử lý dữ liệu thô đầu vào
stage_assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
stage_scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
stage_dt = DecisionTreeClassifier(featuresCol="features", labelCol="high_tip", maxDepth=5, seed=42)

# 2. Ghép nối thành một Pipeline hoàn chỉnh
pipeline = Pipeline(stages=[stage_assembler, stage_scaler, stage_dt])

# 3. Huấn luyện toàn bộ Pipeline trên dữ liệu thô ban đầu (chưa qua VectorAssembler hay Scaler)
raw_train, raw_test = df_labeled.randomSplit([0.8, 0.2], seed=42)
pipeline_model = pipeline.fit(raw_train)

# 4. Áp dụng Pipeline để dự đoán trên dữ liệu kiểm thử thô
pipeline_predictions = pipeline_model.transform(raw_test)
pipeline_predictions.select("trip_distance", "fare_amount", "high_tip", "prediction").show(5)

Bài tập 7: Hãy thay thế giai đoạn stage_dt trong Pipeline bằng LogisticRegression và chạy lại quy trình thử nghiệm.

9 - Lưu trữ & Tái sử dụng mô hình (Model Persistence)

In [ ]:
import shutil
from pyspark.ml import PipelineModel

# Đường dẫn lưu mô hình
model_directory = "./nyc_taxi_final_pipeline"

# Xóa thư mục nếu đã tồn tại
shutil.rmtree(model_directory, ignore_errors=True)

# 1. Lưu Pipeline Model đã huấn luyện xuống đĩa
pipeline_model.save(model_directory)
print(f"Đã lưu thành công Pipeline Model tại thư mục: '{model_directory}'")

# 2. Nạp (Load) lại mô hình từ đĩa lên môi trường làm việc
loaded_model = PipelineModel.load(model_directory)

# 3. Sử dụng mô hình đã nạp để dự đoán dữ liệu mới phát sinh
sample_new_data = raw_test.limit(5)
new_predictions = loaded_model.transform(sample_new_data)

print("\nKết quả dự đoán từ mô hình vừa nạp lại từ ổ đĩa:")
new_predictions.select("trip_distance", "fare_amount", "high_tip", "prediction", "probability").show(truncate=False)

Bài tập 8: Tạo một PySpark DataFrame mới chứa thông tin của 2 chuyến taxi thực tế giả định:
+ Chuyến 1: trip_distance=1.2, fare_amount=8.5, passenger_count=1, pickup_hour=14, PULocationID=138, DOLocationID=263
+ Chuyến 2: trip_distance=15.5, fare_amount=65.0, passenger_count=2, pickup_hour=20, PULocationID=132, DOLocationID=142

Dùng loaded_model dự đoán xem 2 chuyến đi này có khả năng thuộc nhóm high_tip hay không.

Bài tập 9: Hãy tìm cách cải tiến để các mô hình huấn luyện tốt hơn. So sánh với kết quả các mô hình trước đây và phân tích lý do vì sao cải tiến có tác dụng.